# Do EEG foundation models transfer to the injured brain?

Contribution C2 shows deep networks built for healthy sleep collapse on this cohort
(0.61-0.69 accuracy against ~0.85 on healthy sleepers) while 188 physiologically-grounded
features hold at 0.72. EEG foundation models are pretrained on large healthy-sleep corpora,
so the paper predicts they fail here too. Nobody has tested that.

**Design.** Replace the `eeg_enc` MLP (188 features -> 128-d) with a pretrained EEG encoder
reading raw signal. Everything else is identical: cardiorespiratory stream, fusion, BiLSTM,
the validated bypass, both heads. The encoder is the only variable.

| Arm | EEG path |
|---|---|
| A | 188 engineered features (published baseline) |
| B | CBraMod, **frozen**, linear probe to 128-d |
| C | CBraMod, **fine-tuned** end to end |
| D | CBraMod architecture, **randomly initialised** -- the control |

**Arm D is not optional.** Without it a gain from B or C could be the architecture rather
than the pretraining, which is exactly the claim being tested.

Arms A, B and D are cheap: the frozen encoder is precomputed once into
`data/cbramod_emb/`. Arm C trains a 4.9 M-parameter transformer inside every fold and is
about two orders of magnitude more expensive, so it runs as a separate job (section 6).


In [1]:
import json, os, sys, time
import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'MMNet_research', 'model'))
sys.path.insert(0, os.path.join(REPO, 'MMNet_research', 'foundation'))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import mmnet_core as C
import arms

OUT = os.path.join(REPO, 'MMNet_research', 'results', 'revision', 'runs', 'foundation')
os.makedirs(OUT, exist_ok=True)
SEEDS = [42, 1, 7]
ARMS = {'A_features': 'A', 'B_frozen': 'pretrained', 'D_random': 'random'}
print('subjects:', len(C.SUBS), '| device:', C.DEV)


cwd: D:\proc\isleeps-sleep-staging\MMNet_research\MMNet_Submission\all_codes\notebooks | device: cuda | NVIDIA GeForce RTX 5080


subjects: 99 (SN28 dropped) | epochs: 92,560
stage %: {'W': np.float64(26.7), 'N1': np.float64(10.1), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}
respiratory-event prevalence: 15.7%
EEG-feature counts -> EEG: 112 EOG: 50 EMG: 26
parameters (concat): 773,254
training utilities defined.
subjects: 99 | device: cuda


## 1. Run arms A, B and D

Each condition writes its JSON as soon as it finishes, so a killed session loses at most
one run -- the checkpointing pattern `12_multiseed_ablation.ipynb` uses.


In [2]:
CACHE = os.path.join(OUT, 'arms_frozen.json')
results = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

todo = [(a, s) for a in ARMS for s in SEEDS if '%s|%d' % (a, s) not in results]
print('%d runs remaining' % len(todo))

for name, seed in todo:
    t0 = time.time()
    with arms.arm(C, ARMS[name]) as a:
        r = C.run_10fold(fusion='concat', seed=seed)
    key = '%s|%d' % (name, seed)
    results[key] = {
        'arm': name, 'seed': seed, 'eeg_dim': a.dim,
        'acc':   [f['acc']   for f in r['per_fold']],
        'mf1':   [f['mf1']   for f in r['per_fold']],
        'kappa': [f['kappa'] for f in r['per_fold']],
        'auc':   [f['auc']   for f in r['per_fold']],
        'ap':    [f['ap']    for f in r['per_fold']],
    }
    json.dump(results, open(CACHE, 'w'), indent=1)
    print('%-12s seed %-3d dim %-5d acc %.4f  AUC %.4f  [%.1f min]'
          % (name, seed, a.dim, np.mean(results[key]['acc']),
             np.mean(results[key]['auc']), (time.time() - t0) / 60))


0 runs remaining


## 2. Pooled results

Three seeds x ten folds = 30 values per arm, as the handoff specifies.


In [3]:
def pooled(arm, metric):
    v = [results['%s|%d' % (arm, s)][metric] for s in SEEDS if '%s|%d' % (arm, s) in results]
    return np.array(v).ravel()

print('%-12s %20s %20s %20s' % ('arm', 'accuracy', 'macro-F1', 'resp. AUC'))
print('-' * 76)
for a in ARMS:
    cols = [pooled(a, m) for m in ('acc', 'mf1', 'auc')]
    print('%-12s' % a + ''.join(' %12.4f +- %.4f' % (v.mean(), v.std(ddof=1)) for v in cols))


arm                      accuracy             macro-F1            resp. AUC
----------------------------------------------------------------------------
A_features         0.7275 +- 0.0167       0.6536 +- 0.0178       0.7070 +- 0.0528
B_frozen           0.7197 +- 0.0206       0.6549 +- 0.0270       0.7047 +- 0.0515
D_random           0.6994 +- 0.0253       0.6236 +- 0.0287       0.7050 +- 0.0522


## 3. The comparisons that matter

- B against **A** -- does pretraining beat engineered features?
- B against **D** -- is any gain from *pretraining* or just from the architecture?

Paired Wilcoxon with Holm correction, and TOST against a margin of one fold SD for any
claim of no difference. `holm()` and `tost()` are copied verbatim from
`12_multiseed_ablation.ipynb` rather than rewritten.


In [4]:
from scipy.stats import wilcoxon, ttest_rel

def holm(pvals):
    order = np.argsort(pvals)
    m, adj, run = len(pvals), np.empty(len(pvals)), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * pvals[i])
        adj[i] = min(1.0, run)
    return adj

def tost(diff, margin):
    """Two one-sided tests: is |mean difference| inside +-margin?"""
    p_lo = ttest_rel(diff, np.full_like(diff, -margin)).pvalue / 2 if np.mean(diff) > -margin else 1.0
    p_hi = ttest_rel(diff, np.full_like(diff,  margin)).pvalue / 2 if np.mean(diff) <  margin else 1.0
    return max(p_lo, p_hi)


In [5]:
COMPARISONS = [('B_frozen', 'A_features'), ('B_frozen', 'D_random'),
               ('A_features', 'D_random')]

for metric, label in (('acc', 'staging accuracy'), ('auc', 'respiratory AUC')):
    margin = float(np.std(pooled('A_features', metric), ddof=1))   # one fold SD of the baseline
    names, deltas, raws, tosts = [], [], [], []
    for x, y in COMPARISONS:
        a, b = pooled(x, metric), pooled(y, metric)
        ok = ~(np.isnan(a) | np.isnan(b))
        d = a[ok] - b[ok]
        names.append('%s - %s' % (x, y))
        deltas.append(float(d.mean()))
        raws.append(wilcoxon(a[ok], b[ok]).pvalue)
        tosts.append(tost(d, margin))
    adj = holm(np.array(raws))
    print()
    print('%s   (equivalence margin = one fold SD of arm A = %.4f, n = %d)'
          % (label.upper(), margin, int(np.sum(~np.isnan(pooled('A_features', metric))))))
    print('%-26s %9s %10s %10s %11s  %s' % ('comparison', 'delta', 'p raw', 'p Holm', 'p TOST', 'verdict'))
    print('-' * 86)
    for n, d, pr, pa, pt in zip(names, deltas, raws, adj, tosts):
        verdict = 'REAL EFFECT' if pa < 0.05 else ('equivalent (no effect)' if pt < 0.05 else 'INCONCLUSIVE')
        print('%-26s %+9.4f %10.4f %10.4f %11.4f  %s' % (n, d, pr, pa, pt, verdict))



STAGING ACCURACY   (equivalence margin = one fold SD of arm A = 0.0167, n = 30)
comparison                     delta      p raw     p Holm      p TOST  verdict
--------------------------------------------------------------------------------------
B_frozen - A_features        -0.0078     0.0606     0.0606      0.0081  equivalent (no effect)
B_frozen - D_random          +0.0203     0.0001     0.0001      1.0000  REAL EFFECT
A_features - D_random        +0.0281     0.0000     0.0000      1.0000  REAL EFFECT

RESPIRATORY AUC   (equivalence margin = one fold SD of arm A = 0.0528, n = 30)
comparison                     delta      p raw     p Holm      p TOST  verdict
--------------------------------------------------------------------------------------
B_frozen - A_features        -0.0023     0.7000     1.0000      0.0000  equivalent (no effect)
B_frozen - D_random          -0.0003     0.6408     1.0000      0.0000  equivalent (no effect)
A_features - D_random        +0.0020     0.7611     

## 4. Per-fold values

Reported per fold, not only as means, as the handoff requires.


In [6]:
import csv
csv_path = os.path.join(OUT, 'arms_frozen_per_fold.csv')
with open(csv_path, 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['arm', 'seed', 'fold', 'acc', 'mf1', 'kappa', 'auc', 'ap'])
    for k in sorted(results):
        r = results[k]
        for i in range(len(r['acc'])):
            w.writerow([r['arm'], r['seed'], i] + [r[m][i] for m in ('acc', 'mf1', 'kappa', 'auc', 'ap')])
print('wrote', csv_path)


wrote D:\proc\isleeps-sleep-staging\MMNet_research\results\revision\runs\foundation\arms_frozen_per_fold.csv


## 5. Reading the result

Both outcomes are publishable, which is why the experiment is worth running:

- **B and C fail or barely help** -> the strongest form of C2. Not 'small CNNs collapse'
  but 'even models pretrained on thousands of healthy nights do not transfer to the
  injured brain, while engineered physiological features do.'
- **They work** -> the cohort ceiling is breakable, and the paper becomes about making
  foundation models work on pathological sleep.

The B-vs-D row licenses either claim. If B is not distinguishable from D, the architecture
rather than the pretraining is doing the work and no statement about *pretraining* is
supportable in either direction.


## 6. Arm C -- fine-tuned encoder (not yet run)

Arm C trains CBraMod inside every fold instead of reading a frozen cache. Two things make
it a separate job rather than another row above:

**Cost.** At the measured throughput -- 12 ms per 32-epoch forward -- a batch of 32 windows
is 640 sleep epochs, about 0.7 s with the backward pass. With `windows(tr, L//2)` giving
~8.4k training windows that is ~261 batches, so roughly 3 minutes per training epoch and on
the order of 30 hours for 3 seeds x 10 folds, against ~1 minute per 10-fold for the frozen
arms.

**Memory.** `windows()` materialises every training window and pushes it to the GPU in one
tensor. For 188-d features that is ~126 MB. For raw 7-channel signal the same tensor is
8.4k x 20 x 7 x 3000 x 4 B, about **14 GB**, which does not fit alongside the model on a
16 GB card. Arm C therefore cannot reuse the training loop as written: it needs the raw
signal held per subject (~1.5 GB in float16 for the whole cohort) and windows indexed on the
fly, rather than the pre-materialised overlapping windows the feature path uses.

Until Arm C runs, this notebook supports no claim about fine-tuned transfer. Frozen probing
is the weakest way to use a foundation model, and it is the only way used here.
